This page is dedicated to compare single-chain MCMC and multi-chain MCMC

In [ ]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


In [ ]:
# run those checks if package compatibility is in trouble

# import jax
# import tensorflow_probability as tfp
# import jaxlib

# print("jaxlib:", jaxlib.__version__)
# print("TFP:", tfp.__version__)

# !pip show jax
# !pip show jaxlib
# !pip show blackjax

# import jax.numpy as jnp

# import blackjax

# import tensorflow_probability.substrates.jax as tfp
# import inference_gym.using_jax as gym

# print("JAX:", jax.__version__)
# print("BlackJAX:", blackjax.__version__)
# print("TFP:", tfp.__version__)
# print("ArviZ:", avs.__version__)
# print("Inference Gym imported successfully!")

**Package Import and other Setups**

In [ ]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
drive.mount('/content/drive')
# utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/Coomparison_Utility.py'
# with open(utility_link) as f: exec(f.read())

In [ ]:
def single_chain(warmup_length,sample_length, initialize_fn,randomKey,
                 target_log_prob_fn, init_step_size):
    initial_position = initialize_fn((1,), randomKey)
    warmup = blackjax.chees_adaptation(
        target_log_prob_fn,num_chains=1,
        target_acceptance_rate=0.75)
    optimizer = optax.adam(learning_rate=0.001)
    key_warmup, key_sample = random.split(randomKey)
    (last_states, parameters), _= warmup.run(
            key_warmup,
            initial_position,
            init_step_size,
            optimizer,
            warmup_length
            )
    BJX_dhmc = blackjax.dhmc(target_log_prob_fn, **parameters)
    print(last_states)
    initial_state = BJX_dhmc.init(last_states)

    kernel = jax.jit(BJX_dhmc.step)

    def inference_loop(rng_key, kernel, initial_state,num_samples):

        @jax.jit
        def one_step(state, rng_key):
            state, _ = kernel(rng_key, state)
            return state, state

        keys = jax.random.split(rng_key,num_samples)
        _, states = jax.lax.scan(one_step, initial_state, keys)
        return states

    rng_key, sample_key = jax.random.split(key_sample)
    states = inference_loop(sample_key, kernel, initial_state, sample_length)
    mcmc_samples = states.position

    return mcmc_samples

**Hyperparameter Setups**

In [ ]:
# max_warmup = 1000
# warmup_window = 100

# window_array = np.append(np.repeat(10, 10),
#                       np.repeat(warmup_window, max_warmup // warmup_window - 1))

# warmup_length = np.repeat(10, len(window_array))
# for i in range(len(warmup_length) - 1):
#     warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# # Transition kernel for short regime
# repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**Single Chain MCMC**

In [ ]:
target = gym.targets.VectorModel(
    gym.targets.Banana(),
    flatten_sample_transformations=True
)

num_dimensions = target.event_shape[0]

# print("Target:", type(target))
# print("Dimensions:", num_dimensions)
# print("Event shape:", target.event_shape)
# Get some estimates of the mean and variance.
try:
  mean_est = target.sample_transformations['identity'].ground_truth_mean
except:
  print('no ground truth mean')
  mean_est = (result.all_states[num_warmup:, :]).mean(0).mean(0)
try:
  var_est = target.sample_transformations['identity'].ground_truth_standard_deviation**2
except:
  print('no ground truth std dev')
  var_est = ((result.all_states[num_warmup:, :]**2).mean(0).mean(0) -
             mean_est**2)
mean_benchmark = mean_est
var_benchmark = var_est

In [ ]:
def logdensity(x):
    y = target.default_event_space_bijector(x)
    fldj = target.default_event_space_bijector.forward_log_det_jacobian(x)
    return target.unnormalized_log_prob(y) + fldj
offset = 2.0
initial_step_size = 1.
def initialize(shape, key):
    return (10 * random.normal(key, shape+(num_dimensions,))+ offset)

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)

In [ ]:
samples = single_chain(100,2048,initialize,random.PRNGKey(1),logdensity,initial_step_size)

DynamicHMCState(position=Array([[-0.8943695, -2.1930575]], dtype=float32), logdensity=Array([-4.4509635], dtype=float32), logdensity_grad=Array([[-0.03307085, -0.7829454 ]], dtype=float32), random_generator_arg=Array([100], dtype=int32, weak_type=True))


TypeError: grad requires real- or complex-valued inputs (input dtype that is a sub-dtype of np.inexact), but got int32. If you want to use Boolean- or integer-valued inputs, use vjp or set allow_int to True.